In [0]:
batch_data = [
    (101,"Tamil Nadu",1500.00),
    (102,"Kerala",2200.0),
    (103,"Tamil Nadu",800.0),
    (104,"Karnataka",1800.0),
    (105,"Kerala",1200.0)
]

columns = [
    "order_id","shipping_state","final_amount"
]

batch_df = spark.createDataFrame(batch_data,columns)
display(batch_df)

order_id,shipping_state,final_amount
101,Tamil Nadu,1500.0
102,Kerala,2200.0
103,Tamil Nadu,800.0
104,Karnataka,1800.0
105,Kerala,1200.0


### Step 2: Calculate total revenue by state

In [0]:
from pyspark.sql.functions import sum,count

batch_summary_df = (
    batch_df
    .groupBy("shipping_state")
    .agg(
        sum("final_amount").alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
)

display(batch_summary_df)

shipping_state,total_revenue,total_orders
Tamil Nadu,2300.0,2
Kerala,3400.0,2
Karnataka,1800.0,1


### Step 3: Create a streaming source

In [0]:
stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond",5)
    .load()
)

### Step 4: Process incoming records

In [0]:
from pyspark.sql.functions import col

processed_stream_df = stream_df.select(
    col("timestamp"),
    col("value").alias("order_number")
)

### Step 5: Display the streaming results

In [0]:
display(
    processed_stream_df,
    checkpointLocation = '/Volumes/workspace/default/my_files/checkpoint_batch_vs_stream'
)

Checkpointing to /Volumes/workspace/default/my_files/checkpoint_batch_vs_stream
